# Toonistaan GPU Worker (free GPU)
Har scene ki tasveer -> asli chalta AI video clip.

**Kaggle:** Settings → Accelerator → **GPU T4 x2** (ya P100), Internet **On**. **Colab:** Runtime → Change runtime type → **T4 GPU**.
Phir upar se **Run all**. Aakhir mein `clips.zip` download karke Claude ke diye upload link par daal do.

Engine: **LTX-Video** (Lightricks, diffusers `LTXImageToVideoPipeline`, open-weights licence: $10M se kam kamaai par commercial theek). Memory kam pade to `WIDTH`/`HEIGHT` ghatao (dono 32 se divide hone chahiye).

In [ ]:
WIDTH, HEIGHT = 768, 448   # 32 se divide (kam memory: 704x384 / 512x288 nahi: 288 theek, 512 theek)
FPS = 24
STEPS = 30
GUIDANCE = 3.0
SEED = 7

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'diffusers', 'transformers', 'accelerate',
                'sentencepiece', 'ftfy', 'imageio', 'imageio-ffmpeg'], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NAHI MILA - GPU on karo!')

In [ ]:
JOBS = [{"id": "s01", "prompt": "heavy rain falling, lightning flashes in the clouds, trees swaying in strong wind, slow camera push in. an Indian village at night during a heavy thunderstorm, mud houses with dark windows, a big banyan tree, dark blue clouds, wide shot. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s02", "prompt": "the boy blinks, looks around nervously and pulls the blanket closer, the oil lamp flame flickers. inside a dark mud house at night lit only by a small oil lamp, close-up portrait of a cute 8-year-old Indian boy Chhotu with short messy black hair, big expressive brown eyes, wearing a RED t-shirt looking scared, holding a blanket, warm orange lamp light on his face. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s03", "prompt": "the grandfather smiles warmly, nods gently and speaks, the oil lamp flame flickers softly. inside a mud house at night, close-up portrait of a kind old Indian grandfather, 72 years old, white beard and white moustache, bald head with white hair on sides, white kurta, warm smile smiling kindly, lit by a warm oil lamp, cozy. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s05", "prompt": "the boy's face lights up with amazement, he smiles wide and looks around at floating fireflies. outdoors at night in a village, close-up portrait of a cute 8-year-old Indian boy Chhotu with short messy black hair, big expressive brown eyes, wearing a RED t-shirt looking amazed with a big smile, glowing green light on his face, dark blue background. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s06", "prompt": "the grandfather smiles and slowly points up at the glowing fireflies, fireflies drift around. outdoors at night in a village, medium close-up of a kind old Indian grandfather, 72 years old, white beard and white moustache, bald head with white hair on sides, white kurta, warm smile smiling and pointing up, soft blue moonlight and green glow, dark background. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s07", "prompt": "hundreds of fireflies float and twinkle over the grass, the grass sways in a gentle breeze, slow camera rise. a magical night field in an Indian village filled with hundreds of glowing green fireflies, tall grass, a big tree, starry sky, no people, wide shot. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}, {"id": "s08", "prompt": "the boy smiles bravely, fireflies float around him, his hair moves slightly in the breeze. outdoors at night under stars, close-up portrait of a cute 8-year-old Indian boy Chhotu with short messy black hair, big expressive brown eyes, wearing a RED t-shirt smiling bravely and happily, fireflies glowing around him, blue and green light. smooth cinematic animation, consistent character, 3D animated film", "seconds": 4.0, "image_b64": "..."}]

In [ ]:
import os, io, base64, time, zipfile, torch
from PIL import Image
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
assert WIDTH % 32 == 0 and HEIGHT % 32 == 0, 'WIDTH/HEIGHT 32 se divide hone chahiye'
# T4/P100 par bf16 nahi: float32 (2B model ~8 GB, surakshit); A100/L4 par bf16 (tez)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32
os.makedirs('clips', exist_ok=True)
NEG = 'worst quality, inconsistent motion, blurry, jittery, distorted face, extra limbs, deformed hands, flicker, text, watermark'
pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=dtype)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('model taiyaar', dtype)

In [ ]:
for j in JOBS:
    path = f"clips/{j['id']}.mp4"
    if os.path.exists(path):
        print('pehle se', path); continue
    img = Image.open(io.BytesIO(base64.b64decode(j['image_b64']))).convert('RGB').resize((WIDTH, HEIGHT))
    n = int(j['seconds'] * FPS) // 8 * 8 + 1
    t0 = time.time()
    g = torch.Generator('cuda').manual_seed(SEED)
    try:
        frames = pipe(image=img, prompt=j['prompt'], negative_prompt=NEG, width=WIDTH, height=HEIGHT,
                      num_frames=n, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=g).frames[0]
    except torch.cuda.OutOfMemoryError:
        print('MEMORY KAM:', j['id'], '-> WIDTH/HEIGHT chhota karke dobara chalao'); torch.cuda.empty_cache(); continue
    export_to_video(frames, path, fps=FPS)
    print(j['id'], 'ban gaya', round(time.time() - t0), 'sec')
    torch.cuda.empty_cache()

In [ ]:
with zipfile.ZipFile('clips.zip', 'w') as z:
    for f in sorted(os.listdir('clips')):
        z.write(os.path.join('clips', f), f)
print('clips.zip taiyaar:', round(os.path.getsize('clips.zip') / 1e6, 1), 'MB')
try:
    from google.colab import files; files.download('clips.zip')   # Colab
except Exception:
    print('Kaggle: right side Output -> clips.zip -> Download')